# simscDNAm 模型推理

复用已完成训练的 checkpoint、内部 fit 掩码及联合秩映射参数，生成新的完整矩阵。不会重新训练或重新拟合映射。目标 H5AD 只提供 Subclass 配额。

In [ ]:
from pathlib import Path
import sys
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "model/src/model.py").is_file())
SERVER = sys.platform != "win32"
DATA_ROOT = Path("/home/sccasimp/data/xb/simscDNAm") if SERVER else PROJECT / "data/raw"
OUTPUTS = Path("/home/sccasimp/data/xb/outputs/simscDNAm") if SERVER else PROJECT / "outputs"
DATASET = "liu2023/full"
TRAINED_RUN = OUTPUTS / "model" / DATASET
TRAINING_PATH = OUTPUTS / "preprocessing" / DATASET / "TrainingReal.h5ad"
TARGET_OBS_PATH = OUTPUTS / "preprocessing" / DATASET / "HeldoutReal.h5ad"
OUTPUT_DIR = OUTPUTS / "inference" / DATASET / "simscDNAm"
SEED = 42


In [ ]:
import os, math, uuid, hashlib, json, gc
import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from scipy.stats import rankdata
from scipy.special import ndtri
from threadpoolctl import threadpool_limits
sys.path.insert(0, str(PROJECT))
from model.src.model import LatentDiffusionVAE, load_cvae_model, sample_latent_ddim
from data.training import load_target_labels, _close_backed, _dense_float32
assert Path(sys.prefix).name.lower() == "simscdnam", sys.executable
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(8)


## 生成、训练掩码与映射

算法与训练入口一致；所有区域走同一映射，不在生成后复制区域。映射依赖同类型整批细胞。

In [ ]:
def write_synthetic_h5ad(
    output_path: Path,
    model: LatentDiffusionVAE,
    synthetic_class_indices: np.ndarray,
    synthetic_subclass: np.ndarray,
    var: pd.DataFrame,
    feature_names: np.ndarray,
    sampling_steps: int,
    device: torch.device,
    seed: int,
    row_block: int = 8,
    feature_block: int = 4096,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    n_samples = len(synthetic_subclass)
    obs = pd.DataFrame(
        {"origin": "Synthetic", "generator": "End-to-end conditional latent diffusion VAE", "Subclass": synthetic_subclass},
        index=pd.Index([f"synthetic_{i:06d}" for i in range(1, n_samples + 1)], name="cell_id"),
    )
    partial = output_path.with_name(
        f".{output_path.name}.{os.getpid()}.{uuid.uuid4().hex}.partial"
    )
    shell = ad.AnnData(
        X=sp.csr_matrix((n_samples, len(feature_names)), dtype=np.float32),
        obs=obs,
        var=var.copy(),
    )
    shell.var_names = pd.Index(feature_names.astype(str))
    shell.write_h5ad(partial, compression="gzip")
    model.eval()
    with h5py.File(partial, "r+", locking=False) as handle:
        del handle["X"]
        dataset = handle.create_dataset(
            "X", shape=(n_samples, len(feature_names)), dtype="float32",
            chunks=(min(row_block, n_samples), min(feature_block, len(feature_names))),
            compression="gzip", compression_opts=4, shuffle=True,
        )
        dataset.attrs["encoding-type"] = "array"
        dataset.attrs["encoding-version"] = "0.2.0"
        with torch.no_grad():
            for start in range(0, n_samples, row_block):
                stop = min(start + row_block, n_samples)
                condition = torch.from_numpy(
                    synthetic_class_indices[start:stop]
                ).long().to(device)
                latent = sample_latent_ddim(
                    model, condition, sampling_steps, seed + start
                )
                values = model.decode(latent).cpu().numpy().astype(np.float32)
                dataset[start:stop] = values
    partial.replace(output_path)


def validate_synthetic_h5ad(
    path: Path, feature_names: np.ndarray, expected_count: int
) -> dict[str, float | int]:
    item = ad.read_h5ad(path, backed="r")
    try:
        if item.shape != (expected_count, len(feature_names)):
            raise AssertionError(f"合成数据形状不符合预期： {item.shape}")
        if not np.array_equal(item.var_names.astype(str).to_numpy(), feature_names.astype(str)):
            raise AssertionError("合成数据与输入特征轴不一致")
        minimum, maximum = math.inf, -math.inf
        for start in range(0, item.n_obs, 32):
            block = _dense_float32(item.X[start:min(start + 32, item.n_obs)])
            if not np.isfinite(block).all():
                raise AssertionError("合成数据包含非有限值")
            minimum = min(minimum, float(block.min()))
            maximum = max(maximum, float(block.max()))
        if minimum < 0 or maximum > 1:
            raise AssertionError("合成值超出 [0, 1]")
        return {
            "n_cells": int(item.n_obs), "n_features": int(item.n_vars),
            "minimum": minimum, "maximum": maximum,
        }
    finally:
        _close_backed(item)

def sample_mask_rows(training_labels, fit_rows, candidate_labels, seed):
    rng = np.random.default_rng(seed)
    donors = np.empty(len(candidate_labels), dtype=int)
    for level in np.unique(candidate_labels):
        pool = fit_rows[training_labels[fit_rows] == level]
        if not len(pool):
            raise ValueError(f'训练掩码没有细胞类型：{level}')
        rows = np.flatnonzero(candidate_labels == level)
        donors[rows] = rng.choice(pool, len(rows), replace=True)
    return donors

def apply_training_mask(candidate, donor_values):
    if candidate.shape != donor_values.shape:
        raise ValueError('生成矩阵和训练掩码形状不一致')
    return np.where(np.isfinite(donor_values), candidate, np.nan).astype(np.float32)


def write_training_mask_h5ad(raw_path, output_path, matrix, training_labels, fit_rows,
                             seed=42, feature_block=4096):
    """同类型fit细胞整行缺失掩码；所有有限生成值和特征轴保持原样。"""
    raw_path, output_path = Path(raw_path), Path(output_path)
    if output_path.exists():
        raise FileExistsError(f'输出已存在：{output_path}')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    source = ad.read_h5ad(raw_path, backed='r')
    partial = output_path.with_name(output_path.name + '.' + uuid.uuid4().hex + '.partial')
    try:
        if matrix.shape[1] != source.n_vars or len(training_labels) != matrix.shape[0]:
            raise ValueError('训练掩码与生成矩阵的区域轴或行标签不一致')
        labels = source.obs['Subclass'].astype(str).to_numpy()
        donors = sample_mask_rows(np.asarray(training_labels), np.asarray(fit_rows), labels, seed)
        obs = source.obs.copy()
        obs['mask_donor_row'] = donors
        obs['generator'] = 'CVAE＋训练掩码'
        shell = ad.AnnData(sp.csr_matrix(source.shape, dtype=np.float32), obs=obs, var=source.var.copy())
        shell.uns['postprocessing'] = {'method': 'training_mask', 'mask_seed': seed,
            'mask_reference': 'same Subclass, internal fit cells only', 'model_unchanged': True}
        shell.write_h5ad(partial, compression='gzip')
        observed = 0
        with h5py.File(partial, 'r+') as handle:
            del handle['X']
            output = handle.create_dataset('X', shape=source.shape, dtype='float32',
                chunks=(min(8, source.n_obs), min(feature_block, source.n_vars)),
                compression='gzip', compression_opts=1, shuffle=True, fillvalue=np.nan)
            output.attrs['encoding-type'], output.attrs['encoding-version'] = 'array', '0.2.0'
            for start in range(0, source.n_vars, feature_block):
                stop = min(start + feature_block, source.n_vars)
                candidate = source.X[:, start:stop]
                candidate = np.asarray(candidate.toarray() if hasattr(candidate, 'toarray') else candidate, dtype=np.float32)
                donor = np.asarray(matrix[np.ix_(donors, np.arange(start, stop))], dtype=np.float32)
                values = apply_training_mask(candidate, donor)
                np.testing.assert_array_equal(np.isfinite(values), np.isfinite(donor))
                np.testing.assert_array_equal(values[np.isfinite(values)], candidate[np.isfinite(values)])
                observed += int(np.isfinite(values).sum())
                output[:, start:stop] = values
        partial.replace(output_path)
        summary = {'方法': 'CVAE_training_mask', '数据': str(output_path), '掩码种子': seed,
            '细胞数': source.n_obs, '区域数': source.n_vars, '观测率': observed / np.prod(source.shape),
            '仅使用同类型内部训练掩码': True, '有限生成值未改变': True}
        (output_path.parent/'mask_summary.json').write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
        return summary
    finally:
        source.file.close()
        if partial.exists():
            partial.unlink()


def gaussian_ranks(values):
    """有限值转平均秩正态分数，缺失中心填0；各非恒定列方差标准化。"""
    finite=np.isfinite(values)
    counts=finite.sum(0)
    ranks=rankdata(np.where(finite,values,np.nan),axis=0,method='average',nan_policy='omit')
    quantiles=(ranks-0.5)/np.maximum(counts,1)
    scores=ndtri(np.clip(quantiles,1e-6,1-1e-6))
    return standardized_scores(scores,finite)


def standardized_scores(values,observed):
    counts=observed.sum(0)
    mean=np.divide(np.sum(values,where=observed,axis=0,dtype=np.float64),counts,
                   out=np.zeros(values.shape[1]),where=counts>0)
    centered=np.where(observed,values-mean,0)
    deviation=np.sqrt(np.sum(centered**2,axis=0)/max(len(values)-1,1))
    result=np.divide(centered,deviation,out=np.zeros_like(centered),where=deviation>1e-7)
    return result.astype(np.float32)


def symmetric_power(matrix,power):
    values,vectors=np.linalg.eigh((matrix+matrix.T)/2)
    floor=max(float(values.max())*1e-5,1e-8)
    return (vectors*np.maximum(values,floor)**power)@vectors.T


def shared_rank_projection(source_scores,observed,model,n_components):
    q=min(n_components,model['basis'].shape[1],len(source_scores)-1)
    basis=model['basis'][:,:q]
    coordinates=(source_scores-model['mean'])@basis
    coordinates=coordinates@np.linalg.pinv(basis.T@basis,rcond=1e-5)
    coordinates-=coordinates.mean(0)
    covariance=coordinates.T.astype(np.float64)@coordinates.astype(np.float64)/max(len(coordinates)-1,1)
    transform=symmetric_power(covariance,-0.5)@symmetric_power(model['covariance'][:q,:q],0.5)
    transported=(coordinates@transform).astype(np.float32)
    shared=transported@basis.T+model['mean']
    return standardized_scores(shared,observed)


def quantile_from_sorted(reference_sorted,reference_counts,scores,original):
    """逐列显式经验分位数匹配；模型分数只决定细胞间排序。"""
    observed=np.isfinite(original)
    counts=observed.sum(0)
    eligible=(reference_counts>0)&(counts>0)
    result=original.copy()
    if not eligible.any():
        return result
    ranks=rankdata(np.where(observed[:,eligible],scores[:,eligible],np.nan),axis=0,
                   method='average',nan_policy='omit')
    quantiles=(ranks-0.5)/counts[eligible]
    positions=np.nan_to_num(quantiles,nan=0.5)*reference_counts[eligible]-0.5
    positions=np.clip(positions,0,reference_counts[eligible]-1)
    lower=np.floor(positions).astype(int)
    upper=np.minimum(lower+1,reference_counts[eligible]-1)
    ordered=reference_sorted[:,eligible]
    left=np.take_along_axis(ordered,lower,axis=0)
    right=np.take_along_axis(ordered,upper,axis=0)
    mapped=left+(right-left)*(positions-lower)
    result[:,eligible]=np.where(observed[:,eligible],mapped,np.nan)
    return result


def map_class(original,reference_sorted,reference_counts,model,n_components,eta):
    if len(original)<3:
        return original.copy()
    source=gaussian_ranks(original)
    observed=np.isfinite(original)
    if eta:
        shared=shared_rank_projection(source,observed,model,n_components)
    output=np.empty_like(original,dtype=np.float32)
    for start in range(0,original.shape[1],4096):
        stop=min(start+4096,original.shape[1])
        score=source[:,start:stop] if not eta else (1-eta)*source[:,start:stop]+eta*shared[:,start:stop]
        output[:,start:stop]=quantile_from_sorted(reference_sorted[:,start:stop],reference_counts[start:stop],
            score,original[:,start:stop])
    assert np.array_equal(np.isnan(output),np.isnan(original))
    return output


def write_json(path,payload):
    """完成标记和运行签名在写完后原子替换。"""
    path=Path(path)
    temporary=path.with_name(path.name+'.partial')
    temporary.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
    temporary.replace(path)


def feature_axis_hash(names):
    return hashlib.sha256(json.dumps(list(map(str,names)),ensure_ascii=False).encode()).hexdigest()


def apply_joint_mapping(masked_path,output_path,models_dir):
    """固定联合投影强度1并匹配fit经验分位数；所有列遵循同一规则。"""
    masked_path,output_path,models_dir=Path(masked_path),Path(output_path),Path(models_dir)
    if output_path.exists():
        raise FileExistsError(f'输出已存在：{output_path}')
    metadata_path=models_dir/'metadata.json'
    metadata=json.loads(metadata_path.read_text(encoding='utf-8'))
    original=ad.read_h5ad(masked_path)
    if feature_axis_hash(original.var_names)!=metadata['区域轴SHA256']:
        raise ValueError('生成数据的区域名称或顺序与映射不一致')
    values=original.X.toarray() if sp.issparse(original.X) else original.X.copy()
    labels=original.obs['Subclass'].astype(str).to_numpy()
    details=[]
    for record in metadata['类型明细']:
        rows=np.flatnonzero(labels==record['细胞类型'])
        if record['状态']!='已拟合' or len(rows)<3:
            continue
        prefix=models_dir/record['文件前缀']
        with np.load(prefix.with_suffix('.npz')) as saved:
            model={key:saved[key].copy() for key in ['basis','covariance','mean']}
            counts=saved['reference_counts'].copy()
        ordered=np.load(prefix.with_name(prefix.name+'_sorted.npy'),mmap_mode='r')
        with threadpool_limits(limits=8):
            values[rows]=map_class(values[rows],ordered,counts,model,metadata['因子上限'],1.)
        details.append({'细胞类型':record['细胞类型'],'映射细胞数':len(rows)})
        del model,ordered
        gc.collect()
        print('联合秩映射应用：',details[-1],flush=True)
    source_values=original.X.toarray() if sp.issparse(original.X) else original.X
    np.testing.assert_array_equal(np.isnan(values),np.isnan(source_values))
    if np.nanmin(values)<0 or np.nanmax(values)>1:
        raise ValueError('主模型生成值超出0–1')
    result=ad.AnnData(values,obs=original.obs.copy(),var=original.var.copy(),uns=dict(original.uns))
    result.obs['generator']='simscDNAm'
    result.uns['simscDNAm']={'方法':'simscDNAm','版本':'1.0',
        '流程':'CVAE + training mask + joint rank mapping','因子上限':metadata['因子上限'],
        '投影强度':1.,'fit细胞数':len(metadata['fit行']),
        '映射参数SHA256':hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
        '未使用复制配对重写输出':True}
    output_path.parent.mkdir(parents=True,exist_ok=True)
    partial=output_path.with_suffix('.h5ad.partial')
    result.write_h5ad(partial,compression='gzip',compression_opts=1)
    partial.replace(output_path)
    report={'方法':'simscDNAm','状态':'完成','总细胞数':len(labels),
            '映射细胞数':sum(item['映射细胞数'] for item in details),'类型明细':details,
            '数据':str(output_path),'区域数':result.n_vars,'NaN位置保持':True}
    write_json(output_path.parent/'alignment_summary.json',report)
    return report

In [ ]:
if SERVER:
    OUTPUT_DIR.resolve().relative_to(OUTPUTS.resolve())
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError("推理输出目录非空，请使用新的 OUTPUT_DIR")
model_root = TRAINED_RUN / "simscDNAm"
if not (TRAINED_RUN / "completed.json").is_file():
    raise ValueError("训练尚未完成，不能运行正式推理")
training_report = json.loads((TRAINED_RUN / "completed.json").read_text(encoding="utf-8"))
CACHE_RUN = Path(training_report.get("训练缓存目录", str(TRAINED_RUN)))
model, checkpoint = load_cvae_model(model_root / "model.pt", DEVICE)
training = ad.read_h5ad(TRAINING_PATH, backed="r")
matrix = None
try:
    matrix = np.load(CACHE_RUN / "work/values.float32.npy", mmap_mode="r")
    audit = pd.read_csv(CACHE_RUN / "work/input_cell_audit.csv")
    if (training.shape != matrix.shape
            or training.n_vars != checkpoint["input_dim"]
            or feature_axis_hash(training.var_names) != json.loads((model_root / "alignment/metadata.json").read_text(encoding="utf-8"))["区域轴SHA256"]
            or not np.array_equal(training.obs_names.astype(str), audit["cell_id"].astype(str))
            or not np.array_equal(training.obs["Subclass"].astype(str), audit["Subclass"].astype(str))):
        raise ValueError("训练输入与 checkpoint、缓存细胞或标签不一致")
    for start in range(0, training.n_obs, 32):
        if not np.array_equal(_dense_float32(training.X[start:start+32]), matrix[start:start+32], equal_nan=True):
            raise ValueError("当前训练输入与 checkpoint 原训练缓存不同")
    with np.load(CACHE_RUN / "internal_reference.npz") as reference:
        fit_rows = reference["fit_rows"].copy()
    names = training.var_names.to_numpy(dtype=str)
    labels = load_target_labels(TARGET_OBS_PATH, names, checkpoint["class_names"])
    np.random.default_rng(SEED).shuffle(labels)
    class_lookup = {str(name): index for index, name in enumerate(checkpoint["class_names"])}
    indices = np.array([class_lookup[label] for label in labels], dtype=np.int64)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    raw_path, masked_path = OUTPUT_DIR / "raw.h5ad", OUTPUT_DIR / "masked.h5ad"
    write_synthetic_h5ad(raw_path, model, indices, labels, training.var, names,
        checkpoint["sampling_steps"], DEVICE, SEED)
    validate_synthetic_h5ad(raw_path, names, len(labels))
    write_training_mask_h5ad(raw_path, masked_path, matrix,
        training.obs["Subclass"].astype(str).to_numpy(), fit_rows, SEED)
    apply_joint_mapping(masked_path, OUTPUT_DIR / "data.h5ad", model_root / "alignment")
    write_json(OUTPUT_DIR / "run_summary.json", {"状态": "完成", "方法": "simscDNAm",
        "训练目录": str(TRAINED_RUN), "目标配额来源": str(TARGET_OBS_PATH),
        "生成细胞数": len(labels), "数据": str(OUTPUT_DIR / "data.h5ad"), "种子": SEED})
    print("推理完成：", OUTPUT_DIR / "data.h5ad")
finally:
    training.file.close()
    if matrix is not None:
        matrix._mmap.close()
